# For time-lapse monitoring of a fault zone at depth, which fiber installation wins — and where?

SAFOD main hole, June 2026: 989 GPS-timed accelerated-weight-drop shots over 24 h, recorded
simultaneously on **two fiber installations in the same borehole**:

| | cemented | wireline |
|---|---|---|
| construction | 0.9 mm steel tube cemented between casing strings (Lellouch et al. 2019) | deployed on wireline |
| interrogator | Sintela | OptaSense |
| channel spacing | 1.266 m | 2.042 m |
| gauge length | 16.46 m | 10.21 m |
| extent | 864 m, end loop failed at 800 m | turnaround at 3475 m along fiber |

Same borehole, same shots, same wavefield — the only variable is how the fiber is coupled to
the rock. That is a controlled coupling experiment, and the question it answers is which
installation you should pay for if you want to monitor a fault at depth.

**Every figure below earns its place against that question. Nothing else is included.**

| Figure | Role |
|---|---|
| 1 | The wavefield both fibers see — is the experiment valid, and what is the medium? |
| 2 | Detectability vs depth for each fiber — where does each stop seeing the source? |
| 3 | **Repeatability vs depth for each fiber — the crossover is the result** |
| 4 | What each fiber could actually monitor at Parkfield |

### Standing caveats
- **Depth is distance along fiber.** The cemented fiber's interrogator-to-wellhead lead-in is
  unknown; the wireline array begins 3678 m along its fiber (`StartLocusIndex = 1800`,
  corroborated independently by `Output Channel Start = 36000` CSU at n = 1.4682). Fig 1
  estimates the wireline offset from its own moveout intercept; it is not externally verified.
- **Amplitudes are not compared between fibers.** Cemented delivers strain *rate*, wireline
  delivers strain, and the gauge lengths differ by 1.6×. We differentiate the wireline to
  reconcile units, but gauge-length bias remains, so the comparison is built on **correlation
  and timing**, which are insensitive to both.
- **Cemented data are truncated at 800 m**, the published failure depth. Whether the end loop
  was repaired before 2026 is unknown; an ambient-noise test was inconclusive and an OTDR is
  required to settle it.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt

FIG_DIR = '/home/groups/ettore88/nberrios/safod_das_git/notebooks/figures/awd_2026'
OUT_DIR = os.path.join(FIG_DIR, 'scec')
os.makedirs(OUT_DIR, exist_ok=True)

# Acquisition constants, all read from file metadata rather than assumed.
DX_CEM,  GL_CEM  = 1.26606202, 16.4588     # Sintela .pb acquisition_stats
DX_WIRE, GL_WIRE = 2.0419,     10.2095     # OptaSense .h5 /Acquisition attrs
FS, PRE_S = 1000.0, 0.5                    # both fibers: 10 kHz ping / 10 -> 1 kHz

# Published bounds for the cemented fiber (Lellouch et al. 2019, 10.1029/2019JB017533)
FIBER_END_M, FAILURE_M = 864.0, 800.0

BAND = (20.0, 50.0)
Z_MIN, Z_MAX = 130.0, 530.0                # cemented trust region: alluvium -> SNR=3

CEM_C, WIRE_C = 'C0', 'C1'                 # one colour per fiber, used everywhere
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.titlesize': 10.5, 'font.size': 9})

def bandpass(x, lo=BAND[0], hi=BAND[1], fs=FS):
    return sosfiltfilt(butter(4, [lo, hi], btype='band', fs=fs, output='sos'), x, axis=-1)

print('constants loaded')

In [ ]:
# Weighted stack of both fibers. The stored stacks are per-epoch means over the drops
# BOTH fibers captured, so the two are shot-for-shot identical -- that is what makes this
# a controlled comparison rather than two separate surveys.
d = np.load(os.path.join(FIG_DIR, 'epoch_stacks_paired.npz'))
n_common = d['n_common']
good = n_common > 0
w = n_common[good].astype(float)
N_DROPS = int(w.sum())
i0 = int(PRE_S * FS)

cem = np.tensordot(w, d['nano_stacks'][good], axes=(0, 0)) / w.sum()
wire_strain = np.tensordot(w, d['deep_stacks'][good], axes=(0, 0)) / w.sum()

# Units: OptaSense stores strain, Sintela stores strain rate, and readFile_HDF only
# differentiates with diff=True (which paired_stack_job.py does not pass). Left alone the
# two differ by a factor of omega. Differentiating here is equivalent and costs nothing.
wire = np.gradient(wire_strain, 1.0 / FS, axis=-1)

cem_bp, wire_bp = bandpass(cem), bandpass(wire)
z_cem = np.arange(cem.shape[0]) * DX_CEM
z_wire = np.arange(wire.shape[0]) * DX_WIRE

print(f'{good.sum()} epochs, {N_DROPS} paired drops')
print(f'cemented {cem.shape}, spans 0-{z_cem[-1]:.0f} m along fiber')
print(f'wireline {wire.shape}, spans 0-{z_wire[-1]:.0f} m along fiber (array-relative)')

## Figure 1 &mdash; The wavefield both fibers see

Velocity is measured by **slant-stack semblance**, not by picking. Two first-break pickers were
tried and both locked onto the wrong arrival — one onto a late coda, one onto ~1100 m/s energy
that in a fluid-filled borehole is a tube wave. Semblance sums along every trial moveout and
reports which velocities carry coherent energy, so competing arrivals appear as separate peaks
instead of silently capturing the pick.

Running it on **both** fibers does double duty: it confirms they see the same medium, and the
wireline's intercept $t_0$ estimates its depth offset as $t_0 \times V$ — the registration the
metadata leaves unknown.

In [ ]:
def semblance(sec, zrel, i_ref, vgrid, dtgrid, win_s=0.040, fs=FS):
    """Semblance over (velocity, intercept): coherent energy / total energy."""
    nw, nt = int(win_s * fs), sec.shape[1]
    out = np.zeros((vgrid.size, dtgrid.size))
    for iv, v in enumerate(vgrid):
        shifts = (zrel / v * fs).astype(int)
        for it, dt in enumerate(dtgrid):
            idx = i_ref + int(dt * fs) + shifts
            if idx.min() < 0 or idx.max() + nw >= nt:
                continue
            g = np.stack([sec[k, i:i + nw] for k, i in enumerate(idx)])
            den = zrel.size * np.sum(g ** 2)
            out[iv, it] = np.sum(np.sum(g, axis=0) ** 2) / den if den > 0 else 0.0
    return out

V_GRID = np.arange(400.0, 6000.0, 25.0)
T0_GRID = np.arange(0.0, 0.40, 0.002)

def measure(sec_bp, z, zlo, zhi, dx):
    a, b = int(zlo / dx), int(min(zhi, z[-1]) / dx)
    s = semblance(sec_bp[a:b + 1], z[a:b + 1] - zlo, i0, V_GRID, T0_GRID)
    iv, it = np.unravel_index(np.argmax(s), s.shape)
    return s, V_GRID[iv], T0_GRID[it], s[iv, it], (a, b)

# The two fibers do not carry signal over the same interval -- that asymmetry is the
# thesis, not a nuisance. Searching the wireline over the cemented fiber's window returns
# semblance 0.075 (i.e. nothing), because the wireline's coherent energy sits deeper.
# Each fiber is therefore searched over the interval where its own repeatability holds up.
CEM_WIN  = (Z_MIN, Z_MAX)      # 130-530 m
WIRE_WIN = (400.0, 900.0)      # where wireline CC is 0.74 (Fig 3)

sm_c, V_CEM, T0_CEM, S_CEM, (ac, bc) = measure(cem_bp, z_cem, *CEM_WIN, DX_CEM)
sm_w, V_WIRE, T0_WIRE, S_WIRE, (aw, bw) = measure(wire_bp, z_wire, *WIRE_WIN, DX_WIRE)

S_MIN = 0.25    # below this, a semblance peak is not a detection

print(f'cemented : V = {V_CEM:.0f} m/s, t0 = {T0_CEM*1e3:5.1f} ms, semblance {S_CEM:.3f}')
print(f'wireline : V = {V_WIRE:.0f} m/s, t0 = {T0_WIRE*1e3:5.1f} ms, semblance {S_WIRE:.3f}')

# An intercept is only interpretable if the peak it came from is a real detection.
WIRE_OK = S_WIRE >= S_MIN
Z0_CEM = T0_CEM * V_CEM
print(f'\ncemented: implied depth at window top = {CEM_WIN[0] + Z0_CEM:.0f} m '
      f'(window starts at {CEM_WIN[0]:.0f} m)')
if WIRE_OK:
    Z0_WIRE = T0_WIRE * V_WIRE
    WIRE_OFFSET = (WIRE_WIN[0] + Z0_WIRE) - (CEM_WIN[0] + Z0_CEM)
    print(f'wireline: implied depth at window top = {WIRE_WIN[0] + Z0_WIRE:.0f} m')
    print(f'=> wireline runs ~{WIRE_OFFSET:+.0f} m deep relative to cemented')
else:
    WIRE_OFFSET = np.nan
    print(f'wireline: semblance {S_WIRE:.3f} < {S_MIN} -- NO coherent arrival resolved.')
    print('   No velocity, and no depth offset, is claimed for the wireline fiber.')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4.8))

for k, (sec, z, a, b, V, T0, name, col) in enumerate([
        (cem_bp, z_cem, ac, bc, V_CEM, T0_CEM, 'cemented', CEM_C),
        (wire_bp, z_wire, aw, bw, V_WIRE, T0_WIRE, 'wireline', WIRE_C)]):
    s, zz = sec[a:b + 1], z[a:b + 1]
    clip = np.percentile(np.abs(s), 99)
    ax[k].imshow(s.T, aspect='auto', cmap='gray_r',
                 extent=[zz[0], zz[-1], (s.shape[1] - i0) / FS, -PRE_S],
                 vmin=-clip, vmax=clip)
    ax[k].plot(zz, T0 + (zz - zz[0]) / V, col, lw=1.8, label=f'{V:.0f} m/s')
    ax[k].set(ylim=(0.45, -0.05), xlabel='distance along fiber (m)',
              ylabel='time after drop (s)' if k == 0 else '',
              title=f'{"AB"[k]}  {name}, {N_DROPS} drops')
    ax[k].legend(loc='lower right')
    ax[k].grid(False)

for s, V, name, col in [(sm_c, V_CEM, 'cemented', CEM_C), (sm_w, V_WIRE, 'wireline', WIRE_C)]:
    ax[2].plot(V_GRID, s.max(axis=1), col, lw=1.8, label=f'{name} — {V:.0f} m/s')
ax[2].axhline(S_MIN, ls=':', c='0.5', lw=1)
ax[2].set(xlabel='trial velocity (m/s)', ylabel='peak semblance',
          title='C  Cemented resolves the arrival; wireline does not'
                if not WIRE_OK else 'C  Both fibers resolve the same arrival')
ax[2].legend()

fig.suptitle('Fig 1 — Same borehole, same shots, same wavefield: the comparison is controlled',
             fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig1_wavefield.png'), bbox_inches='tight')
plt.show()

## Figure 2 &mdash; Where does each fiber stop seeing the source?

SNR per channel, with the signal window **following the moveout** rather than fixed — a fixed
window would compare the arrival at shallow depth against empty record at depth and
manufacture a decay that isn't there. Noise is the pre-drop window on the same channel, so
each fiber is measured against its own noise floor and the differing units cancel.

In [ ]:
SIG_S = 0.10

def snr_profile(sec_bp, z, V, t0, smooth_m, dx):
    """Moveout-following SNR per channel, smoothed over +/- smooth_m of fiber."""
    n = int(SIG_S * FS)
    sig = np.full(z.size, np.nan)
    noi = np.full(z.size, np.nan)
    for c in range(z.size):
        a = i0 + int((t0 + (z[c] - Z_MIN) / V) * FS)
        if a < 0 or a + n > sec_bp.shape[1] or i0 - n < 0:
            continue
        sig[c] = np.sqrt(np.mean(sec_bp[c, a:a + n] ** 2))
        noi[c] = np.sqrt(np.mean(sec_bp[c, i0 - n:i0] ** 2))
    k = max(1, int(smooth_m / dx))
    box = np.ones(k) / k
    return np.convolve(sig, box, 'same') / np.convolve(noi, box, 'same')

snr_c = snr_profile(cem_bp, z_cem, V_CEM, T0_CEM, 20.0, DX_CEM)
snr_w = snr_profile(wire_bp, z_wire, V_WIRE, T0_WIRE, 20.0, DX_WIRE)

def crossing(z, s, thr=3.0, zmin=130.0, run_m=40.0):
    """Deepest point above threshold that STAYS above it.

    A bare first-crossing test returns the first noise dip -- it put both fibers at
    ~130 m, the very top of the search. Require the curve to hold above `thr` over a
    run of `run_m` before accepting the point."""
    m = (z > zmin) & np.isfinite(s)
    zz, ss = z[m], s[m]
    if zz.size < 2:
        return np.nan
    k = max(1, int(run_m / np.median(np.diff(zz))))
    ok = np.convolve((ss > thr).astype(float), np.ones(k) / k, 'same') > 0.5
    return zz[ok][-1] if ok.any() else np.nan

ZC_CEM, ZC_WIRE = crossing(z_cem, snr_c), crossing(z_wire, snr_w)
print(f'SNR falls below 3 at: cemented {ZC_CEM:.0f} m, wireline {ZC_WIRE:.0f} m')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
mc, mw = z_cem <= FAILURE_M, z_wire <= z_wire.max()
ax.semilogx(snr_c[mc], z_cem[mc], CEM_C, lw=2, label='cemented')
ax.semilogx(snr_w[mw], z_wire[mw], WIRE_C, lw=2, label='wireline')
ax.axvline(3, ls='--', c='0.4', lw=1.2, label='SNR = 3')
ax.axhspan(FAILURE_M, FIBER_END_M, color='0.85', alpha=0.6)
ax.text(ax.get_xlim()[0] * 1.2, (FAILURE_M + FIBER_END_M) / 2,
        'cemented fiber end zone', fontsize=7, va='center', color='0.35')
for zc, col in [(ZC_CEM, CEM_C), (ZC_WIRE, WIRE_C)]:
    if np.isfinite(zc):
        ax.axhline(zc, color=col, ls=':', lw=1.2)
ax.invert_yaxis()
ax.set(xlabel='direct-P SNR', ylabel='distance along fiber (m)',
       title=f'Fig 2 — Detectability vs depth\n'
             f'SNR = 3 at {ZC_CEM:.0f} m (cemented), {ZC_WIRE:.0f} m (wireline)')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig2_detectability.png'), bbox_inches='tight')
plt.show()

## Figure 3 &mdash; Repeatability vs depth &mdash; the core result

Detectability says whether you see the shot once. **Repeatability says whether you can measure
a change** — which is the whole point of monitoring. These are drop-to-drop metrics within
bursts, loaded from the saved products using their **stored** `scan_depth` axes (the scan
channels are subsampled across the fiber, so `arange(n)*dx` selects the wrong ones).

Correlation and timing are used deliberately: unlike amplitude, neither is affected by the
strain/strain-rate difference or the 1.6× gauge-length mismatch.

In [ ]:
rc = np.load(os.path.join(FIG_DIR, 'within_burst_repeatability_shallow_20_50Hz.npz'))
rw = np.load(os.path.join(FIG_DIR, 'within_burst_repeatability_deep_20_50Hz.npz'))

zc_s, zw_s = rc['scan_depth'], rw['scan_depth']
keep_c = zc_s <= FAILURE_M                       # published fiber limit
keep_w = zw_s <= 1200.0                          # beyond this the wireline is at CC~0.02
print(f'cemented: {(~keep_c).sum()}/{zc_s.size} scan channels dropped (past {FAILURE_M:.0f} m)')
print(f'wireline: plotted to 1200 m; {(zw_s > 1200).sum()} deeper channels omitted (CC ~ 0.02)')

def band_median(r, z, key, lo, hi):
    m = (z >= lo) & (z <= hi) & np.isfinite(r[key])
    return float(np.median(r[key][m])) if m.sum() else np.nan

ZONES = [(130, 400), (400, 800)]
print(f'\n{"":22}' + ''.join(f'{f"{a}-{b} m":>22}' for a, b in ZONES))
for key, lab in [('cc_med', 'waveform CC'), ('nrms_med', 'NRMS (%)'),
                 ('lag_scatter_med', 'timing (ms)')]:
    for r, z, name in [(rc, zc_s, 'cemented'), (rw, zw_s, 'wireline')]:
        vals = ''.join(f'{band_median(r, z, key, a, b):22.2f}' for a, b in ZONES)
        print(f'{lab:>13} {name:8}' + vals)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 5.5), sharey=True)
panels = [('cc_med', 'waveform CC', 0.7, False),
          ('nrms_med', 'NRMS (%)', 100.0, False),
          ('lag_scatter_med', 'timing scatter (ms)', 1.0, True)]

for k, (key, lab, thr, logx) in enumerate(panels):
    ax[k].plot(rc[key][keep_c], zc_s[keep_c], CEM_C, lw=2, label='cemented')
    ax[k].plot(rw[key][keep_w], zw_s[keep_w], WIRE_C, lw=2, label='wireline')
    ax[k].axvline(thr, ls='--', c='0.4', lw=1.2)
    ax[k].axhspan(FAILURE_M, FIBER_END_M, color='0.85', alpha=0.6)
    if logx:
        ax[k].set_xscale('log')
    ax[k].set(xlabel=lab, title='ABC'[k] + '  ' + lab)
ax[0].set(ylabel='distance along fiber (m)')
ax[0].set_ylim(1200, 100)
ax[0].legend(loc='lower left')

cc_c = band_median(rc, zc_s, 'cc_med', 130, 400)
cc_w = band_median(rw, zw_s, 'cc_med', 400, 800)
fig.suptitle(f'Fig 3 — Cemented wins shallow (CC {cc_c:.2f} over 130–400 m); '
             f'wireline holds deeper (CC {cc_w:.2f} over 400–800 m)', fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig3_repeatability.png'), bbox_inches='tight')
plt.show()

## Figure 4 &mdash; What could each fiber actually monitor?

Timing repeatability converts to a velocity-change floor: a scatter $\delta t$ against a
traveltime $t = z/V_P$ gives $\delta v/v \approx \delta t / t$. Computing it this way covers
**both** fibers from the same measurement, which the one existing dv/v product cannot do.

The cemented fiber's dedicated dv/v product is overlaid as a check. It lands **below** the
derived curve because it applies CC > 0.7 gating and estimates stretch over a window rather
than a single lag — so treat the derived floor as a conservative upper bound.

> **The literature dv/v ranges below are placeholders and must be checked against the sources
> before this is shown.** Brenguier et al. (2008, *Science*) is the direct Parkfield
> comparison — same fault, same area — and its coseismic value should be read off that paper.

In [ ]:
dv = np.load(os.path.join(FIG_DIR, 'shallow_direct_p_dvv_20_50Hz_locked.npz'))
EPS_MEAS = float(dv['eps_scatter_clean'])
TARGET = float(dv['defazio_target'])
print(f"cemented dv/v product: {EPS_MEAS*100:.3f}% over "
      f"{float(dv['z_min_m']):.0f}-{float(dv['z_max_m']):.0f} m  (target {TARGET*100:.2f}%)")

def dvv_floor(r, z, keep, V):
    """delta_t / t, with t = z / V."""
    zz, dt = z[keep], r['lag_scatter_med'][keep] * 1e-3
    return zz, dt / (zz / V)

zf_c, fl_c = dvv_floor(rc, zc_s, keep_c, V_CEM)
# Same medium, so use the cemented velocity for the wireline when its own is unresolved.
zf_w, fl_w = dvv_floor(rw, zw_s, keep_w, V_WIRE if WIRE_OK else V_CEM)
best_c, best_w = np.nanmin(fl_c), np.nanmin(fl_w)
print(f'derived best floor: cemented {best_c*100:.3f}%, wireline {best_w*100:.3f}%')

# TODO(verify against sources before printing)
SIGNALS = [('Earth tides', 1e-5, 1e-4),
           ('Seasonal / hydrologic', 3e-4, 2e-3),
           ('Coseismic drop, Parkfield', 3e-4, 3e-3),
           ('Large coseismic drop', 3e-3, 1e-2)]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1, 1.25]})

ax[0].semilogx(fl_c * 100, zf_c, CEM_C, lw=2, label='cemented')
ax[0].semilogx(fl_w * 100, zf_w, WIRE_C, lw=2, label='wireline')
ax[0].plot(EPS_MEAS * 100, 250, 'k*', ms=14, label=f'measured, cemented ({EPS_MEAS*100:.2f}%)')
ax[0].axhspan(FAILURE_M, FIBER_END_M, color='0.85', alpha=0.6)
ax[0].set_ylim(1200, 100)
ax[0].set(xlabel='dv/v floor (%)', ylabel='distance along fiber (m)',
          title='A  Smallest detectable velocity change')
ax[0].legend(fontsize=8, loc='lower right')

floor = min(best_c, best_w)
for i, (name, lo, hi) in enumerate(SIGNALS):
    ok = hi >= floor
    ax[1].barh(i, (hi - lo) * 100, left=lo * 100, height=0.5,
               color='C2' if ok else '0.75', edgecolor='k', lw=0.6)
ax[1].axvline(best_c * 100, color=CEM_C, lw=2, label=f'cemented best {best_c*100:.2f}%')
ax[1].axvline(best_w * 100, color=WIRE_C, lw=2, label=f'wireline best {best_w*100:.2f}%')
ax[1].axvline(TARGET * 100, color='k', ls='--', lw=1.4, label=f'target {TARGET*100:.2f}%')
ax[1].set(xscale='log', yticks=range(len(SIGNALS)),
          yticklabels=[s[0] for s in SIGNALS], xlabel='|dv/v| (%)',
          title='B  Against dv/v signals at Parkfield')
ax[1].set_xlim(5e-4, 3)
ax[1].legend(fontsize=8, loc='lower right')

fig.suptitle('Fig 4 — What each installation could monitor', fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig4_monitoring.png'), bbox_inches='tight')
plt.show()

## The answer

One table. If a claim is not here, this notebook does not support it.

In [ ]:
rows = [
    ('Medium velocity (cemented)',
     f'{V_CEM:.0f} m/s', f'semblance {S_CEM:.2f}'),
    ('Medium velocity (wireline)',
     f'{V_WIRE:.0f} m/s' if WIRE_OK else 'not resolved',
     f'semblance {S_WIRE:.2f}' + ('' if WIRE_OK else f' < {S_MIN}')),
    ('Detectability limit (SNR = 3)',
     f'{ZC_CEM:.0f} / {ZC_WIRE:.0f} m', 'cemented / wireline'),
    ('Waveform CC, 130-400 m',
     f"{band_median(rc, zc_s, 'cc_med', 130, 400):.2f} / "
     f"{band_median(rw, zw_s, 'cc_med', 130, 400):.2f}", 'cemented wins shallow'),
    ('Waveform CC, 400-800 m',
     f"{band_median(rc, zc_s, 'cc_med', 400, 800):.2f} / "
     f"{band_median(rw, zw_s, 'cc_med', 400, 800):.2f}", 'wireline holds deeper'),
    ('Timing scatter, 130-400 m',
     f"{band_median(rc, zc_s, 'lag_scatter_med', 130, 400):.2f} / "
     f"{band_median(rw, zw_s, 'lag_scatter_med', 130, 400):.2f} ms", 'cemented / wireline'),
    ('dv/v floor, best',
     f'{best_c*100:.3f} / {best_w*100:.3f} %', f'measured {EPS_MEAS*100:.3f}% (cemented)'),
]
w0 = max(len(r[0]) for r in rows)
print(f'{N_DROPS} paired weight drops, {BAND[0]:.0f}-{BAND[1]:.0f} Hz\n')
for a, b, c in rows:
    print(f'{a:<{w0}}  {b:>20}   ({c})')

## What this notebook does not claim

- **Absolute depth.** Fig 1 estimates the wireline offset from its moveout intercept; that is
  an internal consistency argument, not an external calibration. An OTDR or wellhead tap test
  is required, and it would close the 800 m question at the same time.
- **Relative amplitude or coupling *efficiency*.** Gauge lengths differ 1.6×, so only
  correlation and timing are compared. "Which couples better" is answered here in the sense
  of *repeatability*, not raw sensitivity.
- **Structure in $V_P(z)$.** Only the whole-window velocity is used, because the depth-resolved
  profile has no error bars yet.
- **Anything below 1200 m on the wireline**, where CC falls to ~0.02.

## Next, in order

1. Verify the Fig 4 literature values against Brenguier et al. (2008) and the hydrologic
   dv/v literature.
2. Bootstrap over the 46 epochs for error bars on Figs 2–4.
3. Get the OTDR.